# 💬 WhatsApp AI Agent
**Full-featured WhatsApp Web automation with human-like behavior.**

### Features:
- 🔐 Login via QR Code (session saved — scan once)
- 📩 Send Text Messages
- 📎 Send Images / Videos / Files
- 👍 React to Messages
- 💬 Reply to Messages
- 📢 Send to Groups
- 📊 Read & Save Unread Messages
- ✅ Auto-Reply Bot
- ⏰ Message Scheduler
- 🔍 Search Contacts

> ⚠️ Scans QR code on first run. Session is saved so you won't scan again.
> Uses `undetected-chromedriver` to avoid WhatsApp bot detection.

In [ ]:
# ============================================================
# CELL 1: Imports & Human Behavior Engine
# ============================================================
import time
import random
import os
import threading
import json
from pathlib import Path
from datetime import datetime, timedelta

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

import colorama
from colorama import Fore, Style
colorama.init()

# ── Human delay utilities ─────────────────────────────────────
def human_delay(mn=1.0, mx=3.5):
    time.sleep(random.uniform(mn, mx))

def long_delay(mn=5.0, mx=15.0):
    t = random.uniform(mn, mx)
    print(f"{Fore.YELLOW}  😴 pause {t:.1f}s{Style.RESET_ALL}")
    time.sleep(t)

def human_type(element, text: str):
    """Type text at human speed character by character."""
    for ch in text:
        element.send_keys(ch)
        time.sleep(random.uniform(0.04, 0.12))
    time.sleep(random.uniform(0.2, 0.6))

def safe_click(driver, el):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'})", el)
    time.sleep(random.uniform(0.2, 0.5))
    ActionChains(driver).move_to_element(el).pause(random.uniform(0.1, 0.3)).click().perform()

def wait(driver, css, timeout=20):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, css))
    )

def wait_click(driver, css, timeout=20):
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, css))
    )

def log(msg, level="info"):
    c = {"info": Fore.GREEN, "warn": Fore.YELLOW, "error": Fore.RED, "data": Fore.MAGENTA}.get(level, Fore.WHITE)
    print(f"{c}[{datetime.now():%H:%M:%S}] {msg}{Style.RESET_ALL}")

log("✅ WhatsApp agent loaded.")

In [ ]:
# ============================================================
# CELL 2: Browser Setup & QR Login
# ============================================================
# WhatsApp Web session is saved in this folder
WA_SESSION_DIR = Path("whatsapp_session").absolute()
WA_SESSION_DIR.mkdir(exist_ok=True)

def create_driver(session_dir: Path):
    opts = uc.ChromeOptions()
    # Keep session data so QR is scanned only once
    opts.add_argument(f"--user-data-dir={session_dir}")
    opts.add_argument("--profile-directory=WA_Profile")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    driver = uc.Chrome(options=opts)
    return driver

def whatsapp_login(driver):
    """Open WhatsApp Web. Scan QR first time; saved session used after."""
    driver.get("https://web.whatsapp.com")
    log("⏳ Waiting for WhatsApp Web to load...")
    log("📱 If first run: scan the QR code with your phone now!", "warn")

    try:
        # Wait until chat list is visible (means logged in)
        WebDriverWait(driver, 120).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='chat-list']"))
        )
        log("✅ WhatsApp Web logged in & ready!", "data")
    except TimeoutException:
        log("⚠️  Timed out. Make sure you scanned the QR code.", "error")

driver = create_driver(WA_SESSION_DIR)
whatsapp_login(driver)

In [ ]:
# ============================================================
# CELL 3: Open Chat by Contact Name / Phone
# ============================================================
def open_chat(contact: str):
    """
    Open a chat with a contact.
    contact: saved name or phone number with country code (e.g. +923001234567)
    """
    log(f"Opening chat with: {contact}")
    # Use WhatsApp Web search
    search_btn = wait_click(driver, "[data-testid='chat-list-search']")
    safe_click(driver, search_btn)
    human_delay(0.5, 1.5)

    search_box = wait(driver, "[data-testid='search-input'], #side [contenteditable='true']")
    human_type(search_box, contact)
    human_delay(1.5, 3)

    # Click first result
    try:
        first_result = wait(driver, "[data-testid='cell-frame-container']")
        safe_click(driver, first_result)
        human_delay(1, 3)
        log(f"✅ Chat with '{contact}' opened.", "data")
    except:
        log(f"Contact '{contact}' not found.", "warn")
        # Press ESC to close search
        search_box.send_keys(Keys.ESCAPE)

# ── Example ──────────────────────────────────────────────────
# open_chat("John Doe")
# open_chat("+923001234567")

In [ ]:
# ============================================================
# CELL 4: Send Text Message
# ============================================================
def send_message(contact: str, message: str):
    """Send a text message to a contact."""
    log(f"Sending message to {contact}...")
    open_chat(contact)
    human_delay(1, 3)

    # Type box
    msg_box = wait(driver, "[data-testid='conversation-compose-box-input'], footer [contenteditable='true']")
    safe_click(driver, msg_box)
    human_delay(0.5, 1)
    human_type(msg_box, message)
    human_delay(0.5, 1.5)

    # Send
    msg_box.send_keys(Keys.RETURN)
    log(f"✅ Message sent to {contact}: '{message[:40]}...'", "data")
    long_delay(3, 8)

def send_message_by_phone(phone: str, message: str):
    """
    Send a message via wa.me link (no need for saved contact).
    phone: number with country code, no + or spaces, e.g. '923001234567'
    """
    log(f"Opening wa.me link for {phone}...")
    driver.get(f"https://web.whatsapp.com/send?phone={phone}&text=")
    human_delay(4, 8)

    msg_box = wait(driver, "[data-testid='conversation-compose-box-input']")
    safe_click(driver, msg_box)
    human_type(msg_box, message)
    human_delay(0.5, 1.5)
    msg_box.send_keys(Keys.RETURN)
    log(f"✅ Message sent to +{phone}", "data")
    long_delay(5, 12)

# ── Example ──────────────────────────────────────────────────
# send_message("John Doe", "Hey! How are you? 😊")
# send_message_by_phone("923001234567", "Hi! This is an automated message.")

In [ ]:
# ============================================================
# CELL 5: Send Image / Video / File
# ============================================================
def send_media(contact: str, file_path: str, caption: str = ""):
    """
    Send an image, video, or file to a contact.
    file_path: absolute path to the file
    """
    log(f"Sending media to {contact}: {file_path}")
    open_chat(contact)
    human_delay(1, 3)

    # Click the attach (clip) button
    attach_btn = wait_click(driver, "[data-testid='attach-image-photos-btn'], [title='Attach']")
    safe_click(driver, attach_btn)
    human_delay(0.5, 1.5)

    # Click 'Photos & Videos' or general file option
    try:
        photos_opt = wait_click(driver, "[data-testid='attach-image-photos-btn'] input[type='file']")
    except:
        photos_opt = driver.find_element(By.CSS_SELECTOR, "input[type='file']")

    photos_opt.send_keys(os.path.abspath(file_path))
    human_delay(2, 5)

    # Add caption if any
    if caption:
        try:
            cap_box = wait(driver, "[data-testid='media-caption-input'], [contenteditable='true'][data-tab='10']")
            human_type(cap_box, caption)
            human_delay(0.5, 1.5)
        except: pass

    # Click Send
    send_btn = wait_click(driver, "[data-testid='send-media-confirm-btn'], span[data-icon='send']")
    safe_click(driver, send_btn)
    human_delay(3, 6)
    log(f"✅ Media sent to {contact}.", "data")
    long_delay(5, 12)

# ── Example ──────────────────────────────────────────────────
# send_media("John Doe", r"C:\images\photo.jpg", caption="Check this out! 🔥")

In [ ]:
# ============================================================
# CELL 6: Read & Save Unread Messages
# ============================================================
def get_unread_chats():
    """Return list of chats with unread messages."""
    log("Checking for unread messages...")
    driver.get("https://web.whatsapp.com")
    human_delay(3, 5)

    unread = driver.find_elements(By.CSS_SELECTOR, "[data-testid='icon-unread-count']")
    chat_names = []
    for badge in unread:
        try:
            parent = badge.find_element(By.XPATH, "../../../../../..")
            name   = parent.find_element(By.CSS_SELECTOR, "span[title]").get_attribute("title")
            count  = badge.text
            chat_names.append({"name": name, "unread": count})
            print(f"  📨 {name}: {count} unread")
        except: pass

    log(f"Found {len(chat_names)} unread chats.", "data")
    return chat_names

def read_messages(contact: str, count: int = 20):
    """Open chat and read recent messages."""
    log(f"Reading messages from {contact}...")
    open_chat(contact)
    human_delay(2, 4)

    msgs = driver.find_elements(By.CSS_SELECTOR, 
        "[data-testid='msg-container'], .message-in, .message-out")
    messages = []
    print(f"\n💬 Last {min(count, len(msgs))} messages with {contact}:")
    for m in msgs[-count:]:
        try:
            text  = m.find_element(By.CSS_SELECTOR, "[data-testid='msg-text'], .selectable-text").text
            mtype = "OUT" if "message-out" in m.get_attribute("class") else " IN"
            print(f"  [{mtype}] {text[:80]}")
            messages.append({"type": mtype, "text": text})
        except: pass

    return messages

# ── Example ──────────────────────────────────────────────────
# get_unread_chats()
# read_messages("John Doe", count=10)

In [ ]:
# ============================================================
# CELL 7: React to Last Message
# ============================================================
EMOJI_REACTIONS = ["👍", "❤️", "😂", "😮", "😢", "🙏"]

def react_to_last_message(contact: str, emoji: str = "👍"):
    """
    React to the last message in a chat.
    emoji: one of 👍 ❤️ 😂 😮 😢 🙏
    """
    log(f"Reacting to last message in {contact} with {emoji}...")
    open_chat(contact)
    human_delay(2, 4)

    try:
        messages = driver.find_elements(By.CSS_SELECTOR, "[data-testid='msg-container']")
        last_msg = messages[-1]

        # Hover to show emoji button
        ActionChains(driver).move_to_element(last_msg).perform()
        human_delay(0.5, 1)

        react_btn = last_msg.find_element(By.CSS_SELECTOR, "[data-testid='msg-reaction-btn']")
        safe_click(driver, react_btn)
        human_delay(0.5, 1)

        # Find correct emoji
        emoji_btns = driver.find_elements(By.CSS_SELECTOR, ".emoji-picker-item, [data-testid^='reactive-emoji']")
        for btn in emoji_btns:
            if emoji in btn.text or emoji in btn.get_attribute("aria-label"):
                safe_click(driver, btn)
                log(f"✅ Reacted with {emoji}.", "data")
                break
    except Exception as e:
        log(f"Reaction failed: {e}", "warn")

    human_delay(2, 4)

# ── Example ──────────────────────────────────────────────────
# react_to_last_message("John Doe", emoji="❤️")

In [ ]:
# ============================================================
# CELL 8: Reply to Specific Message
# ============================================================
def reply_to_last_message(contact: str, reply_text: str):
    """Quote-reply the last message in a chat."""
    log(f"Replying to last message in {contact}...")
    open_chat(contact)
    human_delay(2, 4)

    try:
        messages = driver.find_elements(By.CSS_SELECTOR, "[data-testid='msg-container']")
        last_msg = messages[-1]

        # Hover over message
        ActionChains(driver).move_to_element(last_msg).perform()
        human_delay(0.5, 1)

        # Click dropdown arrow
        dropdown = last_msg.find_element(By.CSS_SELECTOR, "[data-testid='down-context']") 
        safe_click(driver, dropdown)
        human_delay(0.5, 1)

        # Click 'Reply'
        reply_option = wait_click(driver, "[data-testid='mi-msg-reply']")
        safe_click(driver, reply_option)
        human_delay(0.5, 1.5)

        # Type reply
        msg_box = wait(driver, "[data-testid='conversation-compose-box-input']")
        human_type(msg_box, reply_text)
        msg_box.send_keys(Keys.RETURN)
        log(f"✅ Replied: '{reply_text[:40]}...'", "data")
    except Exception as e:
        log(f"Reply failed: {e}", "warn")

    long_delay(3, 8)

# ── Example ──────────────────────────────────────────────────
# reply_to_last_message("John Doe", "Thanks for reaching out! 🙌")

In [ ]:
# ============================================================
# CELL 9: Send to Group
# ============================================================
def send_to_group(group_name: str, message: str):
    """Send a message to a WhatsApp group."""
    log(f"Sending to group: {group_name}")
    # Groups are searched the same way as contacts
    send_message(group_name, message)

def send_bulk_messages(contacts: list, message: str, delay_min: int = 60, delay_max: int = 180):
    """
    Send the same message to multiple contacts with randomized long delays.
    contacts  : list of contact names or phone numbers
    delay_min : minimum seconds between messages
    delay_max : maximum seconds between messages
    """
    log(f"Bulk message to {len(contacts)} contacts...")
    for i, contact in enumerate(contacts):
        log(f"  [{i+1}/{len(contacts)}] → {contact}")
        try:
            send_message(contact, message)
        except Exception as e:
            log(f"  Failed for {contact}: {e}", "warn")

        if i < len(contacts) - 1:
            wait_time = random.uniform(delay_min, delay_max)
            log(f"  ⏳ Waiting {wait_time:.0f}s before next send...", "warn")
            time.sleep(wait_time)

    log("✅ Bulk messages done.", "data")

# ── Example ──────────────────────────────────────────────────
# send_to_group("My Study Group", "Hey team! Meeting at 5pm today 📅")
# send_bulk_messages(["Alice", "Bob", "Charlie"], "Hi! Just checking in 👋", delay_min=45, delay_max=120)

In [ ]:
# ============================================================
# CELL 10: Auto-Reply Bot
# ============================================================
AUTO_REPLY_RULES = {
    "hello"    : "Hello! How can I help you? 😊",
    "hi"       : "Hi there! 👋",
    "price"    : "Please visit our website for pricing info.",
    "available": "Yes, I'm available! When works for you?",
    "thanks"   : "You're welcome! 🙏",
}

auto_reply_running = False

def get_last_incoming_message():
    """Get text of the last incoming message in current chat."""
    msgs = driver.find_elements(By.CSS_SELECTOR, ".message-in .selectable-text")
    return msgs[-1].text.strip().lower() if msgs else ""

def auto_reply_loop(contact: str, rules: dict = AUTO_REPLY_RULES, check_interval: int = 10):
    """
    Watch a chat and auto-reply based on keyword rules.
    Runs in background thread.
    """
    global auto_reply_running
    auto_reply_running = True
    log(f"🤖 Auto-reply bot started for {contact}. Press stop_auto_reply() to stop.", "data")
    open_chat(contact)
    replied_texts = set()

    def _loop():
        while auto_reply_running:
            try:
                last = get_last_incoming_message()
                if last and last not in replied_texts:
                    for keyword, reply in rules.items():
                        if keyword in last:
                            human_delay(2, 5)   # simulate reading time
                            msg_box = wait(driver, "[data-testid='conversation-compose-box-input']")
                            human_type(msg_box, reply)
                            msg_box.send_keys(Keys.RETURN)
                            log(f"🤖 Auto-replied to '{last[:30]}': '{reply}'", "data")
                            replied_texts.add(last)
                            break
            except: pass
            time.sleep(check_interval + random.uniform(-2, 3))

    threading.Thread(target=_loop, daemon=True).start()

def stop_auto_reply():
    global auto_reply_running
    auto_reply_running = False
    log("🛑 Auto-reply bot stopped.")

# ── Example ──────────────────────────────────────────────────
# auto_reply_loop("John Doe")

In [ ]:
# ============================================================
# CELL 11: Message Scheduler
# ============================================================
import schedule as sched

def schedule_message(contact: str, message: str, hour: int, minute: int, days: str = "daily"):
    """
    Schedule a message to be sent at a specific time.
    days: 'daily', 'monday', 'tuesday', etc.
    """
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Scheduled message sending to {contact} at {time_str}")
        send_message(contact, message)

    if days == "daily":
        sched.every().day.at(time_str).do(job)
    elif hasattr(sched.every(), days):
        getattr(sched.every(), days).at(time_str).do(job)
    else:
        sched.every().day.at(time_str).do(job)

    log(f"📅 Message to '{contact}' scheduled for {time_str} ({days}).")

def schedule_media(contact: str, file_path: str, caption: str, hour: int, minute: int):
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Sending media to {contact}")
        send_media(contact, file_path, caption)
    sched.every().day.at(time_str).do(job)
    log(f"📅 Media to '{contact}' scheduled for {time_str}.")

def run_scheduler():
    def _loop():
        while True:
            sched.run_pending()
            time.sleep(30)
    threading.Thread(target=_loop, daemon=True).start()
    log("✅ Scheduler running in background. Keep notebook open.")

# ── Example ──────────────────────────────────────────────────
# schedule_message("Mom", "Good morning! ☀️", hour=8, minute=0)
# run_scheduler()

In [ ]:
# ============================================================
# CELL 12: Search Contacts
# ============================================================
def search_contacts(query: str):
    """Search WhatsApp contacts and list matches."""
    log(f"Searching contacts for: {query}")
    driver.get("https://web.whatsapp.com")
    human_delay(2, 4)

    search_btn = wait_click(driver, "[data-testid='chat-list-search']")
    safe_click(driver, search_btn)
    human_delay(0.5, 1)

    search_box = wait(driver, "[data-testid='search-input'], #side [contenteditable='true']")
    human_type(search_box, query)
    human_delay(2, 3)

    results = driver.find_elements(By.CSS_SELECTOR, "[data-testid='cell-frame-title'] span[title]")
    print(f"\n🔍 Results for '{query}':")
    contacts = []
    for r in results:
        name = r.get_attribute("title")
        print(f"  • {name}")
        contacts.append(name)

    search_box.send_keys(Keys.ESCAPE)
    return contacts

# ── Example ──────────────────────────────────────────────────
# search_contacts("Ahmed")

In [ ]:
# ============================================================
# CELL 13: Cleanup
# ============================================================
def close_browser():
    log("Closing browser. Session saved — no QR scan next time.")
    driver.quit()
    log("✅ Done.")

# close_browser()